# BAN 612 — Data Science & AI Job Market Analysis
## Notebook 1: Data Collection & Cleaning

**Team Members:** Joyce Lin, Kwadwo Amoako, Zhuobin Wen, Miguel Davila

**Objective:** Scrape 500+ job listings across Data Science and AI roles from multiple sources, then clean and organize the data for analysis.

---

### AI Usage Disclosure
As required by the project instructions, we document AI usage throughout this notebook:
- **Claude (Anthropic)** was used to help structure the scraping pipeline, generate boilerplate code for API calls and BeautifulSoup parsing, and assist with data cleaning logic.
- All code was reviewed, tested, and adapted by team members.
---

## 1. Setup & Imports

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
import re
import json
from datetime import datetime, timedelta
from urllib.parse import urljoin, quote_plus
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 20)

# Common headers to avoid 403 errors (per Prof. Wang's notebook)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Define Target Job Titles, Search Terms & Relevance Filter

Per the project instructions, we search two domains:
- **Data Science:** Data Scientist, Data Analyst, Quantitative Analyst, Quantitative Researcher, Business Analyst, Business Intelligence Analyst
- **AI-related:** Machine Learning Engineer, AI Research Scientist, Prompt Engineer, Response Engineer

We apply a **relevance filter at scrape time** to only collect jobs whose titles contain keywords related to these roles. This avoids collecting irrelevant listings (e.g., generic software engineering roles) and keeps the pipeline lean.

In [2]:
# Keywords used to filter job titles at scrape time
RELEVANT_KEYWORDS = [
    'data', 'analyst', 'analytics', 'machine learning', 'ml ',
    'ai ', 'artificial intelligence', 'quantitative', 'quant ',
    'business intelligence', 'prompt engineer', 'deep learning',
    'nlp', 'computer vision', 'bi '
]

def is_relevant(title):
    """Check if a job title matches our target roles."""
    title_lower = title.lower()
    return any(kw in title_lower for kw in RELEVANT_KEYWORDS)

# Search queries to use across sources
SEARCH_QUERIES = [
    "data scientist", "data analyst", "business analyst",
    "machine learning engineer", "AI engineer",
    "quantitative analyst", "business intelligence analyst",
    "prompt engineer", "data engineer"
]

# Counters to track how many irrelevant jobs we skip
skipped_counts = {'RemoteOK': 0, 'LinkedIn': 0, 'WeWorkRemotely': 0, 'Indeed': 0}

print(f"Relevance filter active: {len(RELEVANT_KEYWORDS)} keywords")
print(f"Search queries: {len(SEARCH_QUERIES)}")

Relevance filter active: 15 keywords
Search queries: 9


---
## 3. Source 1: RemoteOK API (Free JSON API)

RemoteOK provides a free public JSON API — no HTML parsing needed. This is the most reliable source.

**Data types collected:** strings, floats, dates, lists (tags)

In [3]:
def scrape_remoteok(tags_list, delay=2):
    """
    Scrape jobs from RemoteOK's free JSON API.
    Only keeps listings whose titles pass the relevance filter.
    """
    all_jobs = []
    seen_ids = set()
    
    for tag in tags_list:
        try:
            url = f"https://remoteok.com/api?tags={tag}"
            resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
            
            if resp.status_code == 200:
                data = resp.json()
                # First item is metadata, skip it
                jobs = data[1:] if len(data) > 1 else []
                
                for job in jobs:
                    job_id = job.get('id', '')
                    if job_id in seen_ids:
                        continue
                    seen_ids.add(job_id)
                    
                    title = job.get('position', '')
                    
                    # Relevance filter — skip non-matching titles
                    if not is_relevant(title):
                        skipped_counts['RemoteOK'] += 1
                        continue
                    
                    all_jobs.append({
                        'title': title,
                        'company': job.get('company', ''),
                        'location': job.get('location', 'Remote'),
                        'salary_min': job.get('salary_min'),
                        'salary_max': job.get('salary_max'),
                        'tags': ', '.join(job.get('tags', [])),
                        'date_posted': job.get('date', ''),
                        'url': job.get('url', ''),
                        'description': job.get('description', '')[:500],
                        'source': 'RemoteOK',
                        'search_tag': tag
                    })
                
                print(f"  [{tag}] Found {len(jobs)} listings, kept relevant → {len(all_jobs)} total")
            else:
                print(f"  [{tag}] HTTP {resp.status_code}")
                
        except Exception as e:
            print(f"  [{tag}] Error: {e}")
        
        time.sleep(delay)
    
    return pd.DataFrame(all_jobs)

print("RemoteOK scraper ready.")

RemoteOK scraper ready.


In [4]:
# Run RemoteOK scraper
remoteok_tags = [
    'data-science', 'data-analyst', 'machine-learning',
    'ai', 'artificial-intelligence', 'deep-learning',
    'nlp', 'data-engineer', 'business-analyst',
    'python', 'analytics', 'quantitative',
    'scientist', 'analyst', 'llm', 'generative-ai',
    'data', 'ml', 'business-intelligence'
]

print("Scraping RemoteOK...")
df_remoteok = scrape_remoteok(remoteok_tags)
print(f"\nTotal RemoteOK jobs kept: {len(df_remoteok)}")
print(f"Skipped (irrelevant): {skipped_counts['RemoteOK']}")
df_remoteok.head()

Scraping RemoteOK...
  [data-science] Found 0 listings, kept relevant → 0 total
  [data-analyst] Found 0 listings, kept relevant → 0 total
  [machine-learning] Found 0 listings, kept relevant → 0 total
  [ai] Found 98 listings, kept relevant → 29 total
  [artificial-intelligence] Found 0 listings, kept relevant → 29 total
  [deep-learning] Found 0 listings, kept relevant → 29 total
  [nlp] Found 0 listings, kept relevant → 29 total
  [data-engineer] Found 0 listings, kept relevant → 29 total
  [business-analyst] Found 0 listings, kept relevant → 29 total
  [python] Found 69 listings, kept relevant → 63 total
  [analytics] Found 97 listings, kept relevant → 92 total
  [quantitative] Found 0 listings, kept relevant → 92 total
  [scientist] Found 0 listings, kept relevant → 92 total
  [analyst] Found 85 listings, kept relevant → 149 total
  [llm] Found 0 listings, kept relevant → 149 total
  [generative-ai] Found 0 listings, kept relevant → 149 total
  [data] Found 0 listings, kept releva

,title,company,location,salary_min,salary_max,tags,date_posted,url,description,source,search_tag
0,Data Entry Coordinator,Profound Research,Remote,0,0,"system, training, coordinator, support, testing, growth, travel, education, ...",2026-05-07T00:01:09+00:00,https://remoteOK.com/remote-jobs/remote-data-entry-coordinator-profound-rese...,"<p>\n</p><p>\n</p><p><span data-contrast=""none"">About Profound Research</spa...",RemoteOK,ai
1,Mid Senior AI Cinematic Video Editor,EverAI,,60000,100000,"ai, Video Editor, Long-form Video Content, Motion graphic design, Colour cor...",2026-05-06T11:27:14+00:00,https://remoteOK.com/remote-jobs/remote-mid-senior-ai-cinematic-video-editor...,"<div class=""_descriptionText_135ul_202""><h1>Our Vision &amp; Products</h1><p...",RemoteOK,ai
2,Business Analyst,"Contact Government Services, LLC",,0,0,"analyst, design, jira, system, security, training, technical, support, growt...",2026-05-05T08:00:25+00:00,https://remoteOK.com/remote-jobs/remote-business-analyst-contact-government-...,"<p><span style=""font-size: 16px"">Business Analyst</span></p><p>Employment Ty...",RemoteOK,ai
3,3D Modeling & Python Specialist Freelance AI Trainer Project,Invisible Agency,,0,0,"python, 3d, trainer, training",2026-05-05T00:00:09+00:00,https://remoteOK.com/remote-jobs/remote-3d-modeling-python-specialist-freela...,<div><h3><strong>Project Overview</strong></h3>\n<p>We are sourcing independ...,RemoteOK,ai
4,VFX Artist AI Training Vancouver Canada,Prolific Academic Ltd,Vancouver,0,0,"vfx, training, 3d, gaming, unity, technical",2026-05-04T00:00:09+00:00,https://remoteOK.com/remote-jobs/remote-vfx-artist-ai-training-vancouver-can...,"<div><h2 style=""text-align: center;""><strong><em>VFX Artist - AI Training</e...",RemoteOK,ai


---
## 4. Source 2: LinkedIn Public Job Search

LinkedIn's public job search pages (no login required) can be scraped with care. We add delays between requests to avoid rate limiting, as noted in Prof. Wang's notebook.

**Note:** LinkedIn may block requests after too many. If this happens, wait 5–10 minutes and retry with fewer queries.

In [5]:
def scrape_linkedin(search_queries, locations=None, max_pages=3, delay=5):
    """
    Scrape LinkedIn's public job search (no login required).
    Only keeps listings whose titles pass the relevance filter.
    """
    if locations is None:
        locations = [
            "San Francisco Bay Area", "New York", "Los Angeles",
            "Seattle", "Austin", "Chicago", "Boston", "Remote"
        ]
    
    all_jobs = []
    seen_urls = set()
    
    for query in search_queries:
        for location in locations:
            for page in range(max_pages):
                start = page * 25  # LinkedIn paginates by 25
                try:
                    url = (
                        f"https://www.linkedin.com/jobs/search/"
                        f"?keywords={quote_plus(query)}"
                        f"&location={quote_plus(location)}"
                        f"&start={start}"
                    )
                    
                    resp = requests.get(url, headers=HEADERS, timeout=20)
                    
                    if resp.status_code != 200:
                        print(f"  [{query} | {location}] HTTP {resp.status_code} - skipping")
                        break
                    
                    soup = BeautifulSoup(resp.text, 'html.parser')
                    cards = soup.find_all('div', class_='base-card')
                    
                    if not cards:
                        break  # no more results
                    
                    for card in cards:
                        title_el = card.find('h3', class_='base-search-card__title')
                        company_el = card.find('h4', class_='base-search-card__subtitle')
                        loc_el = card.find('span', class_='job-search-card__location')
                        link_el = card.find('a', class_='base-card__full-link')
                        time_el = card.find('time')
                        
                        title_text = title_el.text.strip() if title_el else ''
                        
                        # Relevance filter — skip non-matching titles
                        if not is_relevant(title_text):
                            skipped_counts['LinkedIn'] += 1
                            continue
                        
                        job_url = link_el['href'].split('?')[0] if link_el else ''
                        if job_url in seen_urls:
                            continue
                        seen_urls.add(job_url)
                        
                        # Extract job ID from URL for Pass 2 detail scraping
                        job_id_match = re.search(r'(\d{8,})', job_url)
                        job_id = job_id_match.group(1) if job_id_match else ''
                        
                        all_jobs.append({
                            'title': title_text,
                            'company': company_el.text.strip() if company_el else '',
                            'location': loc_el.text.strip() if loc_el else '',
                            'date_posted': time_el.get('datetime', '') if time_el else '',
                            'url': job_url,
                            'job_id': job_id,
                            'source': 'LinkedIn',
                            'search_query': query,
                            'search_location': location
                        })
                    
                except Exception as e:
                    print(f"  [{query} | {location}] Error: {e}")
                    break
                
                # Random delay to avoid rate limiting
                time.sleep(delay + random.uniform(1, 3))
            
            print(f"  [{query} | {location}] → {len(all_jobs)} total relevant jobs")
    
    return pd.DataFrame(all_jobs)

print("LinkedIn scraper ready.")

LinkedIn scraper ready.


In [6]:
# Run LinkedIn scraper
linkedin_queries = [
    "data scientist", "data analyst", "machine learning engineer",
    "AI engineer", "business analyst", "quantitative analyst",
    "business intelligence analyst", "prompt engineer"
]

linkedin_locations = [
    "San Francisco Bay Area", "New York", "Los Angeles",
    "Seattle", "Austin", "Chicago", "Remote",
    "Boston", "Washington DC", "Atlanta", "Denver", "Dallas"
]

print("Scraping LinkedIn (this may take 15-20 minutes)...")
print("If you get blocked, reduce queries or wait and retry.\n")
df_linkedin = scrape_linkedin(linkedin_queries, linkedin_locations, max_pages=5, delay=4)
print(f"\nTotal LinkedIn jobs kept: {len(df_linkedin)}")
print(f"Skipped (irrelevant): {skipped_counts['LinkedIn']}")
df_linkedin.head()

Scraping LinkedIn (this may take 15-20 minutes)...
If you get blocked, reduce queries or wait and retry.

  [data scientist | San Francisco Bay Area] → 44 total relevant jobs
  [data scientist | New York] → 154 total relevant jobs
  [data scientist | Los Angeles] → 269 total relevant jobs
  [data scientist | Seattle] → 323 total relevant jobs
  [data scientist | Austin] → 335 total relevant jobs
  [data scientist | Chicago] → 435 total relevant jobs
  [data scientist | Remote] → 480 total relevant jobs
  [data scientist | Boston] → 530 total relevant jobs
  [data scientist | Washington DC] → 631 total relevant jobs
  [data scientist | Atlanta] → 687 total relevant jobs
  [data scientist | Denver] → 734 total relevant jobs
  [data scientist | Dallas] → 790 total relevant jobs
  [data analyst | San Francisco Bay Area] → 883 total relevant jobs
  [data analyst | New York] → 981 total relevant jobs
  [data analyst | Los Angeles] → 1038 total relevant jobs
  [data analyst | Seattle] → 1094 

,title,company,location,date_posted,url,job_id,source,search_query,search_location
0,"Machine Learning Engineer, AI",Biohub,"Redwood City, CA",2026-05-02,https://www.linkedin.com/jobs/view/machine-learning-engineer-ai-at-biohub-43...,4398630444,LinkedIn,data scientist,San Francisco Bay Area
1,Machine Learning Engineer Intern,Tinder,"Palo Alto, CA",2026-05-01,https://www.linkedin.com/jobs/view/machine-learning-engineer-intern-at-tinde...,4407965594,LinkedIn,data scientist,San Francisco Bay Area
2,Data Scientist,Verse,"San Francisco, CA",2026-05-07,https://www.linkedin.com/jobs/view/data-scientist-at-verse-4411631804,4411631804,LinkedIn,data scientist,San Francisco Bay Area
3,"Data Scientist, Analytics",Discord,San Francisco Bay Area,2026-05-09,https://www.linkedin.com/jobs/view/data-scientist-analytics-at-discord-43856...,4385660260,LinkedIn,data scientist,San Francisco Bay Area
4,Data Scientist,Google,"San Bruno, CA",2026-05-04,https://www.linkedin.com/jobs/view/data-scientist-at-google-4408913110,4408913110,LinkedIn,data scientist,San Francisco Bay Area


---
## 5. Source 3: WeWorkRemotely

Server-side rendered HTML — straightforward to scrape (per Prof. Wang's notebook). Good for remote-focused roles.

In [7]:
def scrape_weworkremotely():
    """
    Scrape job listings from WeWorkRemotely.
    Only keeps listings whose titles pass the relevance filter.
    """
    categories = [
        "https://weworkremotely.com/categories/remote-back-end-programming-jobs",
        "https://weworkremotely.com/categories/remote-full-stack-programming-jobs",
        "https://weworkremotely.com/categories/remote-data-jobs",
        "https://weworkremotely.com/remote-jobs/search?term=data+scientist",
        "https://weworkremotely.com/remote-jobs/search?term=machine+learning",
        "https://weworkremotely.com/remote-jobs/search?term=data+analyst",
        "https://weworkremotely.com/remote-jobs/search?term=AI",
    ]
    
    all_jobs = []
    seen_links = set()
    
    for cat_url in categories:
        try:
            resp = requests.get(cat_url, headers=HEADERS, timeout=20)
            soup = BeautifulSoup(resp.text, 'html.parser')
            
            for li in soup.find_all('li', class_='feature'):
                link_el = li.find('a', href=True)
                if not link_el:
                    continue
                
                href = link_el.get('href', '')
                if '/remote-jobs/' not in href:
                    continue
                
                full_url = urljoin('https://weworkremotely.com', href)
                if full_url in seen_links:
                    continue
                seen_links.add(full_url)
                
                company_el = li.find('span', class_='company')
                title_el = li.find('span', class_='title')
                region_el = li.find('span', class_='region')
                
                title_text = title_el.text.strip() if title_el else ''
                
                # Relevance filter — skip non-matching titles
                if not is_relevant(title_text):
                    skipped_counts['WeWorkRemotely'] += 1
                    continue
                
                all_jobs.append({
                    'title': title_text,
                    'company': company_el.text.strip() if company_el else '',
                    'location': region_el.text.strip() if region_el else 'Remote',
                    'url': full_url,
                    'source': 'WeWorkRemotely',
                    'search_category': cat_url.split('/')[-1]
                })
            
            print(f"  [{cat_url.split('/')[-1][:40]}] → {len(all_jobs)} total relevant")
            
        except Exception as e:
            print(f"  Error: {e}")
        
        time.sleep(2)
    
    return pd.DataFrame(all_jobs)

print("WeWorkRemotely scraper ready.")

WeWorkRemotely scraper ready.


In [8]:
# Run WeWorkRemotely scraper
print("Scraping WeWorkRemotely...")
df_wwr = scrape_weworkremotely()
print(f"\nTotal WeWorkRemotely jobs kept: {len(df_wwr)}")
print(f"Skipped (irrelevant): {skipped_counts['WeWorkRemotely']}")
if len(df_wwr) > 0:
    display(df_wwr.head())

Scraping WeWorkRemotely...
  [remote-back-end-programming-jobs] → 0 total relevant
  [remote-full-stack-programming-jobs] → 0 total relevant
  [remote-data-jobs] → 0 total relevant
  [search?term=data+scientist] → 0 total relevant
  [search?term=machine+learning] → 0 total relevant
  [search?term=data+analyst] → 0 total relevant
  [search?term=AI] → 0 total relevant

Total WeWorkRemotely jobs kept: 0
Skipped (irrelevant): 2


---
## 5b. Source 3b: Dice.com (Tech-Focused Job Board)

Dice is specifically focused on tech roles — great for DS/AI listings. Server-side rendered HTML, scrape with care and delays.

In [9]:
def scrape_dice(search_queries, locations=None, delay=4):
    """
    Scrape job listings from Dice.com — tech-focused job board.
    Only keeps listings whose titles pass the relevance filter.
    """
    if locations is None:
        locations = [
            "New York, NY", "San Francisco, CA", "Chicago, IL",
            "Seattle, WA", "Austin, TX", "Remote"
        ]
    
    all_jobs = []
    seen_urls = set()
    
    for query in search_queries:
        for location in locations:
            try:
                url = (
                    f"https://www.dice.com/jobs"
                    f"?q={quote_plus(query)}"
                    f"&location={quote_plus(location)}"
                    f"&radius=30&radiusUnit=mi&pageSize=20"
                )
                resp = requests.get(url, headers=HEADERS, timeout=20)
                if resp.status_code != 200:
                    print(f"  [{query} | {location}] HTTP {resp.status_code}")
                    continue
                soup = BeautifulSoup(resp.text, 'html.parser')
                cards = soup.find_all('div', attrs={'data-cy': 'card'})
                if not cards:
                    cards = soup.find_all('div', class_='card-title-link')
                for card in cards:
                    title_el = card.find('a', attrs={'data-cy': 'card-title-link'})
                    if not title_el:
                        title_el = card.find('h5')
                    company_el = card.find('a', attrs={'data-cy': 'search-result-company-name'})
                    loc_el = card.find('span', attrs={'data-cy': 'search-result-location'})
                    title_text = title_el.text.strip() if title_el else ''
                    if not title_text or not is_relevant(title_text):
                        skipped_counts['Dice'] += 1
                        continue
                    job_url = 'https://www.dice.com' + title_el['href'] if title_el and title_el.get('href') else ''
                    if job_url in seen_urls:
                        continue
                    seen_urls.add(job_url)
                    all_jobs.append({
                        'title': title_text,
                        'company': company_el.text.strip() if company_el else '',
                        'location': loc_el.text.strip() if loc_el else location,
                        'url': job_url,
                        'source': 'Dice',
                        'search_query': query,
                        'search_location': location
                    })
                print(f"  [{query} | {location}] → {len(all_jobs)} total relevant jobs")
            except Exception as e:
                print(f"  [{query} | {location}] Error: {e}")
            time.sleep(delay + random.uniform(1, 2))
    return pd.DataFrame(all_jobs)

print("Dice scraper ready.")

Dice scraper ready.


In [10]:
skipped_counts['Dice'] = 0

dice_queries = [
    "data scientist", "data analyst", "machine learning engineer",
    "AI engineer", "business analyst", "quantitative analyst",
    "prompt engineer", "data engineer"
]

print("Scraping Dice.com...")
df_dice = scrape_dice(dice_queries)
print(f"\nTotal Dice jobs kept: {len(df_dice)}")
print(f"Skipped (irrelevant): {skipped_counts['Dice']}")
df_dice.head()

Scraping Dice.com...
  [data scientist | New York, NY] → 0 total relevant jobs
  [data scientist | San Francisco, CA] → 0 total relevant jobs
  [data scientist | Chicago, IL] → 0 total relevant jobs
  [data scientist | Seattle, WA] → 0 total relevant jobs
  [data scientist | Austin, TX] → 0 total relevant jobs
  [data scientist | Remote] → 0 total relevant jobs
  [data analyst | New York, NY] → 0 total relevant jobs
  [data analyst | San Francisco, CA] → 0 total relevant jobs
  [data analyst | Chicago, IL] → 0 total relevant jobs
  [data analyst | Seattle, WA] → 0 total relevant jobs
  [data analyst | Austin, TX] → 0 total relevant jobs
  [data analyst | Remote] → 0 total relevant jobs
  [machine learning engineer | New York, NY] → 0 total relevant jobs
  [machine learning engineer | San Francisco, CA] → 0 total relevant jobs
  [machine learning engineer | Chicago, IL] → 0 total relevant jobs
  [machine learning engineer | Seattle, WA] → 0 total relevant jobs
  [machine learning engine

""


---
## 6. Source 4: Indeed (Backup Source)

Use this if you need more listings to reach 500+. Indeed is more aggressive with blocking, so use generous delays.

**⚠️ Run this only if your total from Sources 1-3 is below 400.**

In [11]:
def scrape_indeed(search_queries, locations=None, max_pages=2, delay=6):
    """
    Scrape job listings from Indeed search results.
    Only keeps listings whose titles pass the relevance filter.
    """
    if locations is None:
        locations = ["San Francisco, CA", "New York, NY", "Remote"]
    
    all_jobs = []
    seen_titles = set()
    
    for query in search_queries:
        for location in locations:
            for page in range(max_pages):
                start = page * 10
                try:
                    url = (
                        f"https://www.indeed.com/jobs"
                        f"?q={quote_plus(query)}"
                        f"&l={quote_plus(location)}"
                        f"&start={start}"
                    )
                    
                    resp = requests.get(url, headers=HEADERS, timeout=20)
                    if resp.status_code != 200:
                        print(f"  [{query} | {location}] HTTP {resp.status_code}")
                        break
                    
                    soup = BeautifulSoup(resp.text, 'html.parser')
                    
                    job_cards = soup.find_all('div', class_='job_seen_beacon')
                    if not job_cards:
                        job_cards = soup.find_all('div', attrs={'data-jk': True})
                    
                    for card in job_cards:
                        title_el = card.find('h2') or card.find('a', attrs={'data-jk': True})
                        company_el = card.find('span', attrs={'data-testid': 'company-name'})
                        if not company_el:
                            company_el = card.find('span', class_='companyName')
                        loc_el = card.find('div', attrs={'data-testid': 'text-location'})
                        if not loc_el:
                            loc_el = card.find('div', class_='companyLocation')
                        salary_el = card.find('div', class_='salary-snippet-container')
                        if not salary_el:
                            salary_el = card.find('div', attrs={'data-testid': 'attribute_snippet_testid'})
                        
                        title_text = title_el.text.strip() if title_el else ''
                        
                        # Relevance filter — skip non-matching titles
                        if not is_relevant(title_text):
                            skipped_counts['Indeed'] += 1
                            continue
                        
                        dedup_key = f"{title_text}|{company_el.text.strip() if company_el else ''}"
                        if dedup_key in seen_titles:
                            continue
                        seen_titles.add(dedup_key)
                        
                        all_jobs.append({
                            'title': title_text,
                            'company': company_el.text.strip() if company_el else '',
                            'location': loc_el.text.strip() if loc_el else '',
                            'salary_text': salary_el.text.strip() if salary_el else '',
                            'source': 'Indeed',
                            'search_query': query,
                            'search_location': location
                        })
                    
                except Exception as e:
                    print(f"  [{query} | {location}] Error: {e}")
                    break
                
                time.sleep(delay + random.uniform(2, 5))
        
            print(f"  [{query} | {location}] → {len(all_jobs)} total relevant")
    
    return pd.DataFrame(all_jobs)

print("Indeed scraper ready (use as backup if needed).")

Indeed scraper ready (use as backup if needed).


In [12]:
print("Scraping Indeed...")
indeed_queries = ["data scientist", "data analyst", "machine learning engineer", "AI engineer",
                  "business analyst", "quantitative analyst", "prompt engineer"]
indeed_locations = ["San Francisco, CA", "New York, NY", "Austin, TX", "Seattle, WA",
                    "Chicago, IL", "Los Angeles, CA", "Remote"]
df_indeed = scrape_indeed(indeed_queries, indeed_locations, max_pages=3, delay=5)
print(f"\nTotal Indeed jobs kept: {len(df_indeed)}")
print(f"Skipped (irrelevant): {skipped_counts['Indeed']}")
df_indeed.head()

Scraping Indeed...
  [data scientist | San Francisco, CA] HTTP 403
  [data scientist | San Francisco, CA] → 0 total relevant
  [data scientist | New York, NY] HTTP 403
  [data scientist | New York, NY] → 0 total relevant
  [data scientist | Austin, TX] HTTP 403
  [data scientist | Austin, TX] → 0 total relevant
  [data scientist | Seattle, WA] HTTP 403
  [data scientist | Seattle, WA] → 0 total relevant
  [data scientist | Chicago, IL] HTTP 403
  [data scientist | Chicago, IL] → 0 total relevant
  [data scientist | Los Angeles, CA] HTTP 403
  [data scientist | Los Angeles, CA] → 0 total relevant
  [data scientist | Remote] HTTP 403
  [data scientist | Remote] → 0 total relevant
  [data analyst | San Francisco, CA] HTTP 403
  [data analyst | San Francisco, CA] → 0 total relevant
  [data analyst | New York, NY] HTTP 403
  [data analyst | New York, NY] → 0 total relevant
  [data analyst | Austin, TX] HTTP 403
  [data analyst | Austin, TX] → 0 total relevant
  [data analyst | Seattle, WA] 

""


---
## 6b. Source 5: Adzuna API (Easy — Free JSON API with Salary Data)

Adzuna provides a free public API returning structured JSON including salary data — which helps address our project's salary coverage gap.
Sign up at https://developer.adzuna.com to get your free app_id and app_key.

In [13]:
def scrape_adzuna(search_queries, locations=None, delay=2):
    """
    Scrape jobs from Adzuna's free public API.
    Free tier: 250 requests/month. Returns salary data.
    Requires free API credentials from https://developer.adzuna.com
    """
    APP_ID = "your_app_id"
    APP_KEY = "your_app_key"

    if locations is None:
        locations = ['us']

    all_jobs = []
    seen_ids = set()

    for query in search_queries:
        for location in locations:
            for page in range(1, 4):
                try:
                    url = (
                        f"https://api.adzuna.com/v1/api/jobs/{location}/search/{page}"
                        f"?app_id={APP_ID}"
                        f"&app_key={APP_KEY}"
                        f"&results_per_page=50"
                        f"&what={quote_plus(query)}"
                        f"&content-type=application/json"
                    )
                    resp = requests.get(url, timeout=20)
                    if resp.status_code != 200:
                        print(f"  [{query} | {location}] HTTP {resp.status_code}")
                        break
                    data = resp.json()
                    jobs = data.get('results', [])
                    if not jobs:
                        break
                    for job in jobs:
                        job_id = job.get('id', '')
                        if job_id in seen_ids:
                            continue
                        seen_ids.add(job_id)
                        title = job.get('title', '')
                        if not is_relevant(title):
                            skipped_counts['Adzuna'] += 1
                            continue
                        all_jobs.append({
                            'title': title,
                            'company': job.get('company', {}).get('display_name', ''),
                            'location': job.get('location', {}).get('display_name', ''),
                            'salary_min': job.get('salary_min'),
                            'salary_max': job.get('salary_max'),
                            'date_posted': job.get('created', ''),
                            'url': job.get('redirect_url', ''),
                            'description': job.get('description', '')[:500],
                            'source': 'Adzuna',
                            'search_query': query
                        })
                    print(f"  [{query} | page {page}] → {len(all_jobs)} total relevant jobs")
                except Exception as e:
                    print(f"  [{query}] Error: {e}")
                    break
                time.sleep(delay + random.uniform(0.5, 1.5))
    return pd.DataFrame(all_jobs)

print("Adzuna scraper ready. Add your API credentials before running.")

Adzuna scraper ready. Add your API credentials before running.


In [14]:
skipped_counts['Adzuna'] = 0

adzuna_queries = [
    "data scientist", "data analyst", "machine learning engineer",
    "AI engineer", "business analyst", "quantitative analyst",
    "prompt engineer", "data engineer"
]

print("Scraping Adzuna...")
df_adzuna = scrape_adzuna(adzuna_queries)
print(f"\nTotal Adzuna jobs kept: {len(df_adzuna)}")
print(f"Skipped (irrelevant): {skipped_counts['Adzuna']}")
df_adzuna.head()

Scraping Adzuna...
  [data scientist | us] HTTP 401
  [data analyst | us] HTTP 401
  [machine learning engineer | us] HTTP 401
  [AI engineer | us] HTTP 401
  [business analyst | us] HTTP 401
  [quantitative analyst | us] HTTP 401
  [prompt engineer | us] HTTP 401
  [data engineer | us] HTTP 401

Total Adzuna jobs kept: 0
Skipped (irrelevant): 0


""


---
## 6c. Source 6: SimplyHired (Medium — HTML Scraping)

SimplyHired is a large job board with less aggressive blocking than Indeed.
Good geographic coverage and includes salary snippets when available.

In [15]:
def scrape_simplyhired(search_queries, locations=None, delay=4):
    """
    Scrape job listings from SimplyHired.
    Less aggressive blocking than Indeed. Includes salary snippets.
    """
    if locations is None:
        locations = [
            'San Francisco, CA', 'New York, NY', 'Chicago, IL',
            'Seattle, WA', 'Austin, TX', 'Los Angeles, CA', 'Remote'
        ]
    all_jobs = []
    seen_urls = set()
    for query in search_queries:
        for location in locations:
            for page in range(0, 3):
                try:
                    url = (
                        f"https://www.simplyhired.com/search"
                        f"?q={quote_plus(query)}"
                        f"&l={quote_plus(location)}"
                        f"&pn={page}"
                    )
                    resp = requests.get(url, headers=HEADERS, timeout=20)
                    if resp.status_code != 200:
                        print(f"  [{query} | {location}] HTTP {resp.status_code}")
                        break
                    soup = BeautifulSoup(resp.text, 'html.parser')
                    cards = soup.find_all('div', class_='SerpJob-jobCard')
                    if not cards:
                        cards = soup.find_all('article', attrs={'data-jobkey': True})
                    if not cards:
                        break
                    for card in cards:
                        title_el = card.find('a', attrs={'data-testid': 'jobTitle'})
                        if not title_el:
                            title_el = card.find('h3')
                        company_el = card.find('span', attrs={'data-testid': 'companyName'})
                        loc_el = card.find('span', attrs={'data-testid': 'searchSerpJobLocation'})
                        salary_el = card.find('p', attrs={'data-testid': 'searchSerpJobSalaryConfirmed'})
                        title_text = title_el.text.strip() if title_el else ''
                        if not title_text or not is_relevant(title_text):
                            skipped_counts['SimplyHired'] += 1
                            continue
                        job_url = 'https://www.simplyhired.com' + title_el.get('href', '') if title_el else ''
                        if job_url in seen_urls:
                            continue
                        seen_urls.add(job_url)
                        all_jobs.append({
                            'title': title_text,
                            'company': company_el.text.strip() if company_el else '',
                            'location': loc_el.text.strip() if loc_el else location,
                            'salary_text': salary_el.text.strip() if salary_el else '',
                            'url': job_url,
                            'source': 'SimplyHired',
                            'search_query': query,
                            'search_location': location
                        })
                    print(f"  [{query} | {location}] → {len(all_jobs)} total relevant jobs")
                except Exception as e:
                    print(f"  [{query} | {location}] Error: {e}")
                    break
                time.sleep(delay + random.uniform(1, 3))
    return pd.DataFrame(all_jobs)

print("SimplyHired scraper ready.")

SimplyHired scraper ready.


In [16]:
skipped_counts['SimplyHired'] = 0

simplyhired_queries = [
    "data scientist", "data analyst", "machine learning engineer",
    "AI engineer", "business analyst", "quantitative analyst",
    "prompt engineer", "data engineer"
]

print("Scraping SimplyHired...")
df_simplyhired = scrape_simplyhired(simplyhired_queries)
print(f"\nTotal SimplyHired jobs kept: {len(df_simplyhired)}")
print(f"Skipped (irrelevant): {skipped_counts['SimplyHired']}")
df_simplyhired.head()

Scraping SimplyHired...
  [data scientist | San Francisco, CA] HTTP 403
  [data scientist | New York, NY] HTTP 403
  [data scientist | Chicago, IL] HTTP 403
  [data scientist | Seattle, WA] HTTP 403
  [data scientist | Austin, TX] HTTP 403
  [data scientist | Los Angeles, CA] HTTP 403
  [data scientist | Remote] HTTP 403
  [data analyst | San Francisco, CA] HTTP 403
  [data analyst | New York, NY] HTTP 403
  [data analyst | Chicago, IL] HTTP 403
  [data analyst | Seattle, WA] HTTP 403
  [data analyst | Austin, TX] HTTP 403
  [data analyst | Los Angeles, CA] HTTP 403
  [data analyst | Remote] HTTP 403
  [machine learning engineer | San Francisco, CA] HTTP 403
  [machine learning engineer | New York, NY] HTTP 403
  [machine learning engineer | Chicago, IL] HTTP 403
  [machine learning engineer | Seattle, WA] HTTP 403
  [machine learning engineer | Austin, TX] HTTP 403
  [machine learning engineer | Los Angeles, CA] HTTP 403
  [machine learning engineer | Remote] HTTP 403
  [AI engineer |

""


---
## 6d. Source 7: Wellfound / AngelList (Hard — Startup-Focused Job Board)

Wellfound (formerly AngelList) lists jobs at startups — a segment not covered by LinkedIn.
AI/ML roles at startups are well represented here. More likely to block than other sources,
so generous delays are used. May require retries if blocked.

In [17]:
def scrape_wellfound(search_queries, delay=5):
    """
    Scrape job listings from Wellfound (formerly AngelList).
    Focuses on startup jobs — covers AI/ML roles not found on LinkedIn.
    Uses generous delays as Wellfound is more aggressive with blocking.
    """
    all_jobs = []
    seen_urls = set()
    for query in search_queries:
        for page in range(1, 4):
            try:
                url = (
                    f"https://wellfound.com/jobs"
                    f"?role={quote_plus(query)}"
                    f"&page={page}"
                )
                resp = requests.get(url, headers=HEADERS, timeout=20)
                if resp.status_code != 200:
                    print(f"  [{query} | page {page}] HTTP {resp.status_code}")
                    break
                soup = BeautifulSoup(resp.text, 'html.parser')
                cards = soup.find_all('div', attrs={'data-test': 'StartupResult'})
                if not cards:
                    cards = soup.find_all('div', class_='styles_component__job')
                if not cards:
                    break
                for card in cards:
                    title_el = card.find('a', attrs={'data-test': 'job-title'})
                    if not title_el:
                        title_el = card.find('span', class_='styles_title__xpQMX')
                    company_el = card.find('a', attrs={'data-test': 'startup-link'})
                    loc_el = card.find('span', attrs={'data-test': 'location'})
                    salary_el = card.find('span', attrs={'data-test': 'compensation'})
                    title_text = title_el.text.strip() if title_el else ''
                    if not title_text or not is_relevant(title_text):
                        skipped_counts['Wellfound'] += 1
                        continue
                    job_url = 'https://wellfound.com' + title_el.get('href', '') if title_el else ''
                    if job_url in seen_urls:
                        continue
                    seen_urls.add(job_url)
                    all_jobs.append({
                        'title': title_text,
                        'company': company_el.text.strip() if company_el else '',
                        'location': loc_el.text.strip() if loc_el else '',
                        'salary_text': salary_el.text.strip() if salary_el else '',
                        'url': job_url,
                        'source': 'Wellfound',
                        'search_query': query
                    })
                print(f"  [{query} | page {page}] → {len(all_jobs)} total relevant jobs")
            except Exception as e:
                print(f"  [{query} | page {page}] Error: {e}")
                break
            time.sleep(delay + random.uniform(2, 4))
    return pd.DataFrame(all_jobs)

print("Wellfound scraper ready.")

Wellfound scraper ready.


In [18]:
skipped_counts['Wellfound'] = 0

wellfound_queries = [
    "data scientist", "data analyst", "machine learning engineer",
    "AI engineer", "business analyst", "quantitative analyst",
    "prompt engineer", "data engineer"
]

print("Scraping Wellfound (startup-focused, use generous delays)...")
df_wellfound = scrape_wellfound(wellfound_queries)
print(f"\nTotal Wellfound jobs kept: {len(df_wellfound)}")
print(f"Skipped (irrelevant): {skipped_counts['Wellfound']}")
df_wellfound.head()

Scraping Wellfound (startup-focused, use generous delays)...
  [data scientist | page 1] HTTP 403
  [data analyst | page 1] HTTP 403
  [machine learning engineer | page 1] HTTP 403
  [AI engineer | page 1] HTTP 403
  [business analyst | page 1] HTTP 403
  [quantitative analyst | page 1] HTTP 403
  [prompt engineer | page 1] HTTP 403
  [data engineer | page 1] HTTP 403

Total Wellfound jobs kept: 0
Skipped (irrelevant): 0


""


---
## 7. Combine All Sources

In [19]:
# Standardize columns across all sources before merging
standard_cols = [
    'title', 'company', 'location', 'salary_min', 'salary_max',
    'salary_text', 'tags', 'date_posted', 'url', 'source', 'description',
    'detail_description', 'detail_seniority', 'detail_employment_type',
    'detail_job_function', 'detail_industry', 'detail_salary',
    'detail_work_arrangement', 'detail_applicants', 'detail_scraped'
]

def standardize_df(df, source_name):
    """Add missing standard columns with NaN."""
    for col in standard_cols:
        if col not in df.columns:
            df[col] = np.nan
    df = df[standard_cols].copy()
    return df

# Combine all DataFrames
frames = []

if len(df_remoteok) > 0:
    frames.append(standardize_df(df_remoteok, 'RemoteOK'))
    
if len(df_linkedin) > 0:
    frames.append(standardize_df(df_linkedin, 'LinkedIn'))
    
if len(df_wwr) > 0:
    frames.append(standardize_df(df_wwr, 'WeWorkRemotely'))

if len(df_dice) > 0:
    frames.append(standardize_df(df_dice, 'Dice'))

if len(df_indeed) > 0:
    frames.append(standardize_df(df_indeed, 'Indeed'))

if len(df_adzuna) > 0:
    frames.append(standardize_df(df_adzuna, 'Adzuna'))

if len(df_simplyhired) > 0:
    frames.append(standardize_df(df_simplyhired, 'SimplyHired'))

if len(df_wellfound) > 0:
    frames.append(standardize_df(df_wellfound, 'Wellfound'))

df_raw = pd.concat(frames, ignore_index=True)

print(f"Total relevant listings collected: {len(df_raw)}")
print(f"\nListings by source:")
print(df_raw['source'].value_counts())
print(f"\nIrrelevant jobs filtered out during scraping:")
for src, count in skipped_counts.items():
    if count > 0:
        print(f"  - {src}: {count} skipped")
print(f"\n{'✅' if len(df_raw) >= 500 else '⚠️'} {'Met' if len(df_raw) >= 500 else 'Below'} 500 listing minimum")

Total relevant listings collected: 4676

Listings by source:
source
LinkedIn    4527
RemoteOK     149
Name: count, dtype: int64

Irrelevant jobs filtered out during scraping:
  - RemoteOK: 169 skipped
  - LinkedIn: 1922 skipped
  - WeWorkRemotely: 2 skipped

✅ Met 500 listing minimum


In [20]:
# Save raw data as checkpoint
df_raw.to_csv('job_listings_raw.csv', index=False)
print(f"Saved {len(df_raw)} raw listings to job_listings_raw.csv")
df_raw.info()

Saved 4676 raw listings to job_listings_raw.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4676 entries, 0 to 4675
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   title                    4676 non-null   object 
 1   company                  4676 non-null   object 
 2   location                 4676 non-null   object 
 3   salary_min               149 non-null    float64
 4   salary_max               149 non-null    float64
 5   salary_text              0 non-null      float64
 6   tags                     149 non-null    object 
 7   date_posted              4676 non-null   object 
 8   url                      4676 non-null   object 
 9   source                   4676 non-null   object 
 10  description              149 non-null    object 
 11  detail_description       0 non-null      float64
 12  detail_seniority         0 non-null      float64
 13  detail_employment_type   0 non

---
## 8. PASS 2: Scrape Full LinkedIn Job Posting Pages

Now we visit each individual LinkedIn job page to extract detailed information not available in search results:
- **Full job description** (responsibilities, requirements, qualifications)
- **Salary/compensation** (when listed on the page or in the description)
- **Seniority level** (Entry, Associate, Mid-Senior, Director, Executive)
- **Employment type** (Full-time, Part-time, Contract, Internship)
- **Industry** (Technology, Finance, Healthcare, etc.)
- **Job function** (Engineering, Analytics, Research, etc.)
- **Work arrangement** (Remote, Hybrid, On-site)

We use LinkedIn's guest job posting endpoint: `https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{JOB_ID}`

**⏱️ This step takes 1-2 hours.** Progress is saved every 50 jobs so you can resume if interrupted.

In [21]:
def scrape_job_detail(job_id):
    """
    Scrape a single LinkedIn job posting page using the guest API endpoint.
    Returns dict with: description, seniority, employment_type, job_function,
    industry, salary, work_arrangement, applicants.
    """
    result = {
        'detail_description': '',
        'detail_seniority': '',
        'detail_employment_type': '',
        'detail_job_function': '',
        'detail_industry': '',
        'detail_salary': '',
        'detail_work_arrangement': '',
        'detail_applicants': '',
        'detail_scraped': False,
    }
    
    if not job_id or str(job_id).strip() == '':
        return result
    
    # Try guest API endpoint, then fallback to regular URL
    for url in [
        f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}",
        f"https://www.linkedin.com/jobs/view/{job_id}/",
    ]:
        try:
            resp = requests.get(url, headers=HEADERS, timeout=20)
            if resp.status_code == 200 and len(resp.text) > 200:
                soup = BeautifulSoup(resp.text, 'html.parser')
                break
        except:
            pass
    else:
        return result  # All URLs failed
    
    # ===== FULL JOB DESCRIPTION =====
    desc_el = soup.find('div', class_='show-more-less-html__markup')
    if not desc_el:
        desc_el = soup.find('div', class_='description__text')
    if not desc_el:
        for div in soup.find_all('div'):
            if 'description' in ' '.join(div.get('class', [])) and len(div.get_text(strip=True)) > 200:
                desc_el = div
                break
    if desc_el:
        result['detail_description'] = desc_el.get_text(separator=' ', strip=True)[:5000]
    
    # ===== JOB CRITERIA (seniority, employment type, function, industry) =====
    for item in soup.find_all('li', class_='description__job-criteria-item'):
        header = item.find('h3')
        value = item.find('span')
        if not header or not value:
            continue
        h = header.get_text(strip=True).lower()
        v = value.get_text(strip=True)
        if 'seniority' in h:
            result['detail_seniority'] = v
        elif 'employment' in h:
            result['detail_employment_type'] = v
        elif 'function' in h:
            result['detail_job_function'] = v
        elif 'industr' in h:
            result['detail_industry'] = v
    
    # ===== SALARY =====
    for tag, cls in [('div','salary'), ('div','compensation__salary'), ('span','compensation__salary')]:
        el = soup.find(tag, class_=cls)
        if el:
            result['detail_salary'] = el.get_text(strip=True)
            break
    if not result['detail_salary']:
        page_text = soup.get_text()
        sal = re.findall(r'\$[\d,]+(?:\.\d+)?(?:\s*[-\u2013\u2014to]+\s*\$[\d,]+(?:\.\d+)?)?(?:\s*(?:per|/|a)?\s*(?:year|yr|hour|hr|annum))?', page_text)
        if sal:
            result['detail_salary'] = max(sal, key=len).strip()
    if not result['detail_salary'] and result['detail_description']:
        sal = re.findall(r'\$[\d,]+(?:\.\d+)?(?:\s*[-\u2013\u2014to]+\s*\$[\d,]+(?:\.\d+)?)', result['detail_description'])
        if sal:
            result['detail_salary'] = max(sal, key=len).strip()
    
    # ===== WORK ARRANGEMENT =====
    for el in soup.find_all(['span', 'li'], class_=re.compile(r'ui-label|workplace', re.I)):
        text = el.get_text(strip=True).lower()
        if 'remote' in text:
            result['detail_work_arrangement'] = 'Remote'; break
        elif 'hybrid' in text:
            result['detail_work_arrangement'] = 'Hybrid'; break
        elif 'on-site' in text or 'onsite' in text:
            result['detail_work_arrangement'] = 'On-site'; break
    if not result['detail_work_arrangement']:
        page_lower = soup.get_text().lower()
        if any(s in page_lower for s in ['fully remote', 'work remotely', 'remote position', '100% remote']):
            result['detail_work_arrangement'] = 'Remote'
        elif 'hybrid' in page_lower:
            result['detail_work_arrangement'] = 'Hybrid'
        elif any(s in page_lower for s in ['on-site only', 'in-office', 'in office required']):
            result['detail_work_arrangement'] = 'On-site'
    
    # ===== APPLICANT COUNT =====
    for cls in ['num-applicants__caption', 'applicant-count']:
        el = soup.find(['span', 'figcaption'], class_=cls)
        if el:
            result['detail_applicants'] = el.get_text(strip=True)
            break
    
    if result['detail_description'] or result['detail_seniority'] or result['detail_industry']:
        result['detail_scraped'] = True
    
    return result

print("Job detail scraper ready.")

Job detail scraper ready.


In [22]:
# Test on 3 jobs before running the full batch
test_ids = df_linkedin[df_linkedin.get('job_id', pd.Series(dtype=str)).astype(str) != '']['job_id'].head(3).tolist() if 'job_id' in df_linkedin.columns else []

if not test_ids:
    print("⚠️ No job IDs found in LinkedIn data. Skipping Pass 2.")
    print("The LinkedIn scraper needs to save 'job_id' — check the scraper output.")
else:
    print("Testing Pass 2 on 3 jobs...\n")
    for tid in test_ids:
        print(f"Job ID: {tid}")
        res = scrape_job_detail(tid)
        for k, v in res.items():
            if k == 'detail_description':
                print(f"  {k}: {str(v)[:120]}..." if v else f"  {k}: (empty)")
            else:
                print(f"  {k}: {v}")
        print()
        time.sleep(3)
    
    if res.get('detail_scraped'):
        print("✅ Detail scraping works! Proceed to the full batch below.")
    else:
        print("⚠️ No data returned. If HTTP 403, LinkedIn may be blocking.")
        print("Try waiting 5 min or using a different network.")

Testing Pass 2 on 3 jobs...

Job ID: 4398630444
  detail_description: Biohub is the first large-scale initiative bringing frontier AI models, massive compute, and frontier experimental capab...
  detail_seniority: Not Applicable
  detail_employment_type: Full-time
  detail_job_function: Engineering and Information Technology
  detail_industry: Biotechnology Research
  detail_salary: $150,000 to $350,000
  detail_work_arrangement: 
  detail_applicants: Over 200 applicants
  detail_scraped: True

Job ID: 4407965594
  detail_description: Our Mission As humans, there are few things more exciting than meeting someone new. At Tinder, we’re inspired by the cha...
  detail_seniority: Not Applicable
  detail_employment_type: Full-time
  detail_job_function: Engineering
  detail_industry: Business Consulting and Services, IT Services and IT Consulting, and Professional Training and Coaching
  detail_salary: $47.00/hr - $47.00/hr
  detail_work_arrangement: Hybrid
  detail_applicants: 66 applicant

In [23]:
# Scrapes every LinkedIn job's individual page for full details.
# Saves progress every 50 jobs. Re-run this cell to resume if interrupted.

import os

CHECKPOINT = 'linkedin_details_checkpoint.csv'

if 'job_id' not in df_linkedin.columns:
    print("⚠️ No job_id column — skipping Pass 2")
else:
    # Init detail columns
    detail_cols = ['detail_description', 'detail_seniority', 'detail_employment_type',
                   'detail_job_function', 'detail_industry', 'detail_salary',
                   'detail_work_arrangement', 'detail_applicants', 'detail_scraped']
    for col in detail_cols:
        if col not in df_linkedin.columns:
            df_linkedin[col] = '' if col != 'detail_scraped' else False
    
    # Load checkpoint if exists
    already_done = set()
    if os.path.exists(CHECKPOINT):
        ckpt = pd.read_csv(CHECKPOINT)
        already_done = set(ckpt[ckpt['detail_scraped'] == True]['job_id'].astype(str))
        for _, row in ckpt[ckpt['detail_scraped'] == True].iterrows():
            mask = df_linkedin['job_id'].astype(str) == str(row['job_id'])
            if mask.any():
                for col in detail_cols:
                    if col in row.index and pd.notna(row[col]):
                        df_linkedin.loc[mask, col] = row[col]
        print(f"Resuming: {len(already_done)} already scraped")
    
    to_scrape = df_linkedin[
        (df_linkedin['job_id'].astype(str) != '') &
        (~df_linkedin['job_id'].astype(str).isin(already_done))
    ].index.tolist()
    
    total = len(to_scrape)
    print(f"Jobs to scrape: {total} (~{total * 4.5 / 60:.0f} min)")
    
    success = failures = consecutive_fails = 0
    
    for i, idx in enumerate(to_scrape):
        jid = str(df_linkedin.loc[idx, 'job_id'])
        details = scrape_job_detail(jid)
        
        for k, v in details.items():
            df_linkedin.loc[idx, k] = v
        
        if details['detail_scraped']:
            success += 1; consecutive_fails = 0
        else:
            failures += 1; consecutive_fails += 1
        
        if (i+1) % 5 == 0:
            print(f"  [{i+1}/{total}] ✓{success} ✗{failures}", end='\r')
        
        if (i+1) % 50 == 0:
            df_linkedin.to_csv(CHECKPOINT, index=False)
            print(f"\n  💾 Checkpoint at {i+1}/{total}")
        
        if consecutive_fails >= 20:
            df_linkedin.to_csv(CHECKPOINT, index=False)
            print(f"\n  ⚠️ Rate limited after {consecutive_fails} failures. Wait 10-15 min, re-run.")
            break
        
        time.sleep(random.uniform(3, 6))
    
    df_linkedin.to_csv(CHECKPOINT, index=False)
    print(f"\n\nPass 2 done! ✓{success} scraped, ✗{failures} failed")

Jobs to scrape: 4527 (~340 min)
  [50/4527] ✓50 ✗0
  💾 Checkpoint at 50/4527
  [100/4527] ✓100 ✗0
  💾 Checkpoint at 100/4527
  [150/4527] ✓150 ✗0
  💾 Checkpoint at 150/4527
  [200/4527] ✓200 ✗0
  💾 Checkpoint at 200/4527
  [250/4527] ✓250 ✗0
  💾 Checkpoint at 250/4527
  [300/4527] ✓300 ✗0
  💾 Checkpoint at 300/4527
  [350/4527] ✓350 ✗0
  💾 Checkpoint at 350/4527
  [400/4527] ✓400 ✗0
  💾 Checkpoint at 400/4527
  [450/4527] ✓450 ✗0
  💾 Checkpoint at 450/4527
  [500/4527] ✓500 ✗0
  💾 Checkpoint at 500/4527
  [550/4527] ✓550 ✗0
  💾 Checkpoint at 550/4527
  [600/4527] ✓600 ✗0
  💾 Checkpoint at 600/4527
  [650/4527] ✓650 ✗0
  💾 Checkpoint at 650/4527
  [700/4527] ✓700 ✗0
  💾 Checkpoint at 700/4527
  [750/4527] ✓750 ✗0
  💾 Checkpoint at 750/4527
  [800/4527] ✓800 ✗0
  💾 Checkpoint at 800/4527
  [850/4527] ✓850 ✗0
  💾 Checkpoint at 850/4527
  [900/4527] ✓900 ✗0
  💾 Checkpoint at 900/4527
  [950/4527] ✓950 ✗0
  💾 Checkpoint at 950/4527
  [1000/4527] ✓999 ✗1
  💾 Checkpoint at 1000/4527
  [1050/4

In [24]:
# Review Pass 2 results
if 'detail_scraped' in df_linkedin.columns:
    print("=" * 60)
    print("PASS 2 RESULTS")
    print("=" * 60)
    print(f"Successfully scraped: {(df_linkedin['detail_scraped'] == True).sum()} / {len(df_linkedin)}")
    print(f"\nField coverage:")
    for col, label in [('detail_description','Description'), ('detail_seniority','Seniority'),
                       ('detail_employment_type','Employment Type'), ('detail_industry','Industry'),
                       ('detail_salary','Salary'), ('detail_work_arrangement','Work Arrangement')]:
        if col in df_linkedin.columns:
            filled = df_linkedin[col].replace('', np.nan).notna().sum()
            print(f"  {label:25s}: {filled:5d} ({filled/len(df_linkedin)*100:.1f}%)")
    
    print(f"\nSeniority levels:")
    print(df_linkedin['detail_seniority'].replace('', np.nan).value_counts())
    print(f"\nWork arrangement:")
    print(df_linkedin['detail_work_arrangement'].replace('', np.nan).value_counts())
    print(f"\nTop industries:")
    print(df_linkedin['detail_industry'].replace('', np.nan).value_counts().head(10))
    print(f"\nSalary samples:")
    sal = df_linkedin[df_linkedin['detail_salary'].replace('', np.nan).notna()]
    if len(sal) > 0:
        print(sal[['title', 'company', 'detail_salary']].head(10).to_string())
    else:
        print("  No salary found on detail pages.")
else:
    print("Pass 2 was skipped (no job_id column).")

PASS 2 RESULTS
Successfully scraped: 4518 / 4527

Field coverage:
  Description              :  4518 (99.8%)
  Seniority                :  4518 (99.8%)
  Employment Type          :  4518 (99.8%)
  Industry                 :  4518 (99.8%)
  Salary                   :  2826 (62.4%)
  Work Arrangement         :  1376 (30.4%)

Seniority levels:
detail_seniority
Not Applicable      2914
Mid-Senior level     959
Associate            381
Entry level          215
Director              26
Internship            18
Executive              5
Name: count, dtype: int64

Work arrangement:
detail_work_arrangement
Hybrid     1013
Remote      265
On-site      98
Name: count, dtype: int64

Top industries:
detail_industry
IT Services and IT Consulting           593
Software Development                    476
Financial Services                      473
Technology, Information and Internet    332
Business Consulting and Services        155
Hospitals and Health Care               105
Staffing and Recruiting  

---
## 9. Data Cleaning & Preparation

Per project instructions: handle missing data, outliers, incorrect records, redundant records, and data type constraints.

In [17]:
# Load from checkpoint (can restart from here)
df = pd.read_csv('job_listings_raw.csv')
print(f"Loaded {len(df)} raw records")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDuplicates (by title + company): {df.duplicated(subset=['title', 'company']).sum()}")

Loaded 4676 raw records

Missing values:
title                         0
company                       0
location                     52
salary_min                 4527
salary_max                 4527
salary_text                4676
tags                       4527
date_posted                   0
url                           0
source                        0
description                4527
detail_description         4676
detail_seniority           4676
detail_employment_type     4676
detail_job_function        4676
detail_industry            4676
detail_salary              4676
detail_work_arrangement    4676
detail_applicants          4676
detail_scraped             4676
dtype: int64

Duplicates (by title + company): 671


In [18]:
# Merge Pass 2 detail data into raw dataset
import os

CHECKPOINT = 'linkedin_details_checkpoint.csv'

if os.path.exists(CHECKPOINT):
    df_details = pd.read_csv(CHECKPOINT)
    
    detail_cols = ['job_id', 'detail_description', 'detail_seniority',
                   'detail_employment_type', 'detail_job_function',
                   'detail_industry', 'detail_salary',
                   'detail_work_arrangement', 'detail_applicants', 'detail_scraped']
    
    # Only keep columns that exist
    detail_cols = [c for c in detail_cols if c in df_details.columns]
    df_details = df_details[detail_cols].copy()
    
    # Match on job_id
    if 'job_id' in df.columns and 'job_id' in df_details.columns:
        df_details['job_id'] = df_details['job_id'].astype(str)
        df['job_id'] = df['job_id'].astype(str)
        df = df.merge(df_details, on='job_id', how='left', suffixes=('', '_det'))
        
        # Use detail columns, drop duplicates
        for col in [c for c in detail_cols if c != 'job_id']:
            if f'{col}_det' in df.columns:
                df[col] = df[col].combine_first(df[f'{col}_det'])
                df.drop(columns=[f'{col}_det'], inplace=True)
        
        scraped_count = df['detail_scraped'].eq(True).sum()
        print(f"✅ Merged Pass 2 data: {scraped_count} jobs with full details")
        print(f"  Description coverage: {df['detail_description'].replace('', None).notna().sum()}")
        print(f"  Salary coverage: {df['detail_salary'].replace('', None).notna().sum()}")
        print(f"  Work arrangement: {df['detail_work_arrangement'].replace('', None).notna().sum()}")
    else:
        print("⚠️ No job_id column found — matching by title+company")
        # Fallback: match on title + company
        df_details_tc = pd.read_csv(CHECKPOINT)
        for _, det_row in df_details_tc[df_details_tc.get('detail_scraped', False) == True].iterrows():
            mask = (df['title'] == det_row.get('title', '')) & (df['company'] == det_row.get('company', ''))
            if mask.any():
                for col in [c for c in detail_cols if c != 'job_id' and c in det_row.index]:
                    df.loc[mask, col] = det_row[col]
        print(f"✅ Merged via title+company match")
else:
    print("⚠️ linkedin_details_checkpoint.csv not found")
    print("  Detail fields will be empty — re-run Pass 2 or check file location")

⚠️ No job_id column found — matching by title+company
✅ Merged via title+company match


### 9.1 Remove Duplicates & Empty Rows

In [19]:
before = len(df)

# Remove exact duplicates
df = df.drop_duplicates(subset=['title', 'company'], keep='first')

# Remove listings with no title
df = df[df['title'].str.strip().ne('') & df['title'].notna()].copy()

print(f"Removed {before - len(df)} duplicate/empty records")
print(f"Remaining: {len(df)}")

Removed 671 duplicate/empty records
Remaining: 4005


### 9.2 Classify Job Titles & Detect AI Usage in Descriptions


In [20]:
def classify_job_title(title):
    """Classify job title into a standard category."""
    t = title.lower()
    if any(kw in t for kw in ['machine learning', 'ml engineer', 'ml ']):
        return 'Machine Learning Engineer'
    if any(kw in t for kw in ['ai engineer', 'ai research', 'artificial intelligence']):
        return 'AI Engineer / Researcher'
    if 'prompt engineer' in t:
        return 'Prompt Engineer'
    if any(kw in t for kw in ['deep learning', 'nlp', 'computer vision', 'cv engineer']):
        return 'ML/AI Specialist'
    if 'data scientist' in t:
        return 'Data Scientist'
    if 'data analyst' in t:
        return 'Data Analyst'
    if any(kw in t for kw in ['data engineer', 'data infrastructure']):
        return 'Data Engineer'
    if any(kw in t for kw in ['quantitative', 'quant ']):
        return 'Quantitative Analyst'
    if any(kw in t for kw in ['business intelligence', 'bi analyst', 'bi developer']):
        return 'Business Intelligence Analyst'
    if 'business analyst' in t:
        return 'Business Analyst'
    if 'analytics' in t:
        return 'Analytics / Other'
    if 'data' in t:
        return 'Data (General)'
    if 'ai' in t:
        return 'AI (General)'
    return 'Other'

# AI keywords to detect in descriptions even when not in the title
AI_DESCRIPTION_KEYWORDS = [
    'machine learning', 'deep learning', 'neural network', 'large language model',
    'llm', 'generative ai', 'gpt', 'transformer', 'nlp', 'natural language processing',
    'computer vision', 'tensorflow', 'pytorch', 'scikit-learn', 'xgboost',
    'reinforcement learning', 'mlops', 'model training', 'model deployment',
    'feature engineering', 'vector database', 'embedding', 'fine-tuning',
    'langchain', 'rag', 'retrieval augmented', 'diffusion model', 'stable diffusion',
    'hugging face', 'openai', 'anthropic', 'ai model', 'ml model', 'ml pipeline'
]

def detect_ai_in_description(row):
    """
    Returns True if the description mentions AI/ML concepts,
    even if the job title doesn't contain AI keywords.
    Uses the full scraped description from Pass 2.
    """
    # Combine all text sources
    text_parts = [
        str(row.get('detail_description', '')),
        str(row.get('description', '')),
    ]
    full_text = ' '.join(text_parts).lower()
    
    if not full_text.strip():
        return False
    
    return any(kw in full_text for kw in AI_DESCRIPTION_KEYWORDS)

def classify_domain(category, ai_in_description=False):
    """
    Classify into AI or Data Science domain.
    Also marks as 'AI (Description)' if the description contains AI keywords
    even though the title category is Data Science.
    """
    ai_cats = ['Machine Learning Engineer', 'AI Engineer / Researcher',
               'Prompt Engineer', 'ML/AI Specialist', 'AI (General)']
    ds_cats = ['Data Scientist', 'Data Analyst', 'Data Engineer',
               'Quantitative Analyst', 'Business Intelligence Analyst',
               'Business Analyst', 'Analytics / Other', 'Data (General)']
    if category in ai_cats:
        return 'AI'
    elif category in ds_cats:
        return 'Data Science'
    return 'Other'

# Apply title classification
df['job_category'] = df['title'].apply(classify_job_title)

# Drop 'Other' titles
other_count = (df['job_category'] == 'Other').sum()
if other_count > 0:
    print(f"Dropping {other_count} 'Other' jobs")
    df = df[df['job_category'] != 'Other'].copy()

# Apply domain classification
df['domain'] = df.apply(lambda row: classify_domain(row['job_category']), axis=1)

# Detect AI usage in descriptions (even for DS-titled roles)
df['ai_in_description'] = df.apply(detect_ai_in_description, axis=1)

# Create a combined AI signal column
# 'AI' = title says AI
# 'AI-Adjacent' = title says DS but description mentions AI heavily
# 'Data Science' = traditional DS role, no AI in description
def get_ai_signal(row):
    if row['domain'] == 'AI':
        return 'AI'
    elif row['domain'] == 'Data Science' and row['ai_in_description']:
        return 'AI-Adjacent'
    return 'Data Science'

df['ai_signal'] = df.apply(get_ai_signal, axis=1)

print("Job category distribution:")
print(df['job_category'].value_counts())
print(f"\nDomain distribution:")
print(df['domain'].value_counts())
print(f"\nAI signal (including description-based detection):")
print(df['ai_signal'].value_counts())
print(f"\nDS roles with AI in description (AI-Adjacent):")
print(df[df['ai_signal'] == 'AI-Adjacent']['job_category'].value_counts())


Dropping 466 'Other' jobs
Job category distribution:
job_category
Data Analyst                     637
Business Analyst                 583
Data Scientist                   524
Machine Learning Engineer        431
AI Engineer / Researcher         264
Business Intelligence Analyst    244
Quantitative Analyst             237
AI (General)                     227
Data (General)                   173
Analytics / Other                112
Data Engineer                     59
Prompt Engineer                   46
ML/AI Specialist                   2
Name: count, dtype: int64

Domain distribution:
domain
Data Science    2569
AI               970
Name: count, dtype: int64

AI signal (including description-based detection):
ai_signal
AI-Adjacent     1535
Data Science    1034
AI               970
Name: count, dtype: int64

DS roles with AI in description (AI-Adjacent):
job_category
Data Scientist                   460
Data Analyst                     343
Business Analyst                 231
Quantit

### 9.3 Clean Location Data

In [21]:
def classify_work_type(row):
    """Determine work arrangement. Priority: scraped detail > title/location keywords."""
    # Priority 1: Use scraped work arrangement from job detail page
    scraped = str(row.get('detail_work_arrangement', '')).strip()
    if scraped and scraped not in ['', 'nan']:
        return scraped
    
    # Priority 2: Check title and location for remote signals
    title = str(row.get('title', '')).lower()
    location = str(row.get('location', '')).lower()
    combined = title + ' ' + location
    
    remote_keywords = ['remote', 'anywhere', 'worldwide', 'work from home', 'wfh']
    if any(kw in combined for kw in remote_keywords):
        return 'Remote'
    if 'hybrid' in combined:
        return 'Hybrid'
    return 'Unknown'

def extract_state(location):
    if pd.isna(location):
        return None
    match = re.search(r',\s*([A-Z]{2})(?:\s|$|,)', str(location))
    if match:
        return match.group(1)
    return None

df['work_type'] = df.apply(classify_work_type, axis=1)
df['state'] = df['location'].apply(extract_state)

print("Work type distribution:")
print(df['work_type'].value_counts())
print(f"\nTop states:")
print(df['state'].value_counts().head(10))


Work type distribution:
work_type
Unknown    2419
Hybrid      801
Remote      248
On-site      71
Name: count, dtype: int64

Top states:
state
CA    591
MA    310
IL    301
TX    285
NY    278
GA    277
NC    246
WA    217
VA    165
MD     93
Name: count, dtype: int64


### 9.4 Parse & Clean Salary Data

Salary data comes from three sources: RemoteOK API (direct), Adzuna API (direct),
and LinkedIn job detail pages (extracted from page content and description text).


In [22]:
def parse_salary_text(text):
    """Extract min and max salary from a salary text string."""
    if pd.isna(text) or str(text).strip() == '':
        return None, None
    
    text = str(text).replace(',', '').replace('$', '')
    
    # Match ranges like "120000 - 180000" or "120K - 180K"
    range_match = re.findall(r'(\d+\.?\d*)\s*[kK]?', text)
    
    if len(range_match) >= 2:
        vals = [float(x) for x in range_match[:2]]
        if 'k' in text.lower():
            vals = [v * 1000 if v < 1000 else v for v in vals]
        if all(v < 500 for v in vals):
            vals = [v * 2080 for v in vals]  # hourly to annual
        return min(vals), max(vals)
    elif len(range_match) == 1:
        val = float(range_match[0])
        if 'k' in text.lower():
            val *= 1000
        if val < 500:
            val *= 2080
        return val, val
    
    return None, None

# Parse salary_text where salary_min/max aren't already set
mask = df['salary_min'].isna() & df['salary_text'].notna()
parsed = df.loc[mask, 'salary_text'].apply(
    lambda x: pd.Series(parse_salary_text(x), index=['sal_min_parsed', 'sal_max_parsed'])
)
if len(parsed) > 0:
    df.loc[mask, 'salary_min'] = parsed['sal_min_parsed'].values
    df.loc[mask, 'salary_max'] = parsed['sal_max_parsed'].values

# Convert to numeric
df['salary_min'] = pd.to_numeric(df['salary_min'], errors='coerce')
df['salary_max'] = pd.to_numeric(df['salary_max'], errors='coerce')

# Calculate midpoint salary
df['salary_mid'] = df[['salary_min', 'salary_max']].mean(axis=1)

# Remove salary outliers (likely errors)
salary_mask = df['salary_mid'].notna()
df.loc[salary_mask & (df['salary_mid'] < 20000), ['salary_min', 'salary_max', 'salary_mid']] = np.nan
df.loc[salary_mask & (df['salary_mid'] > 500000), ['salary_min', 'salary_max', 'salary_mid']] = np.nan

print(f"Jobs with salary data: {df['salary_mid'].notna().sum()} / {len(df)} ({df['salary_mid'].notna().mean():.1%})")
if df['salary_mid'].notna().sum() > 0:
    print(f"\nSalary summary (annual):")
    print(df['salary_mid'].describe().apply(lambda x: f"${x:,.0f}" if pd.notna(x) else x))

Jobs with salary data: 11 / 3539 (0.3%)

Salary summary (annual):
count         $11
mean     $107,727
std       $58,368
min       $45,000
25%       $75,000
50%       $80,000
75%      $135,000
max      $215,000
Name: salary_mid, dtype: object


### 9.5 Extract Features from Descriptions (Education, Experience, Skills)


In [27]:
# ===== EDUCATION REQUIREMENTS =====
EDUCATION_PATTERNS = {
    'PhD': [
        'ph.d', 'phd', 'doctorate', 'doctoral degree', 'doctor of philosophy',
        'ph.d.', 'd.phil', 'postdoc', 'post-doctoral'
    ],
    "Master's": [
        "master's degree", "master's", "m.s.", "ms in", "m.s in",
        "mba", "m.b.a", "graduate degree", "advanced degree",
        "master of science", "master of business", "master of arts",
        "master of engineering", "ms/phd", "bs/ms", "ma in", "m.eng"
    ],
    "Bachelor's": [
        "bachelor's degree", "bachelor's", "b.s.", "bs in", "b.s in",
        "b.a.", "ba in", "undergraduate degree", "4-year degree",
        "bachelor of science", "bachelor of arts", "bachelor of engineering",
        "bsc", "b.eng", "bs/ba", "b.s/b.a"
    ],
    'No Degree Required': [
        "no degree", "degree not required", "equivalent experience",
        "or equivalent", "high school", "self-taught", "bootcamp",
        "without a degree", "experience in lieu"
    ]
}

def extract_education(row):
    text = ' '.join([
        str(row.get('detail_description', '')),
        str(row.get('description', ''))
    ]).lower()
    
    if not text.strip() or text.strip() == 'nan nan':
        return 'Not Specified'
    
    for level, patterns in EDUCATION_PATTERNS.items():
        if any(p in text for p in patterns):
            return level
    return 'Not Specified'

df['education_required'] = df.apply(extract_education, axis=1)

print("Education requirement distribution:")
print(df['education_required'].value_counts())
print(f"\nEducation by domain:")
print(pd.crosstab(df['domain'], df['education_required']))

# ===== YEARS OF EXPERIENCE REQUIRED =====
def extract_years_experience(row):
    text = ' '.join([
        str(row.get('detail_description', '')),
        str(row.get('description', ''))
    ]).lower()
    
    if not text.strip() or text.strip() == 'nan nan':
        return None
    
    patterns = [
        r'(\d+)\+\s*years?\s+(?:of\s+)?(?:experience|work)',
        r'(\d+)\s*[-–]\s*\d+\s*years?\s+(?:of\s+)?(?:experience|work)',
        r'minimum\s+(?:of\s+)?(\d+)\s*years?',
        r'at\s+least\s+(\d+)\s*years?',
        r'(\d+)\s*years?\s+(?:of\s+)?(?:professional\s+|relevant\s+|industry\s+|work\s+)?experience',
        r'(\d+)\s*to\s*\d+\s*years?\s+(?:of\s+)?experience',
        r'(\d+)\+\s*yrs',
        r'(\d+)\s*yrs?\s+(?:of\s+)?experience',
        r'experience[:\s]+(\d+)\+?\s*years?',
    ]
    
    found = []
    for pattern in patterns:
        matches = re.findall(pattern, text)
        found.extend([int(m) for m in matches if str(m).isdigit() and int(m) <= 20])
    
    if found:
        return min(found)
    return None

df['min_years_exp'] = df.apply(extract_years_experience, axis=1)

print(f"\nYears experience coverage: {df['min_years_exp'].notna().sum()} / {len(df)}")
print(f"\nYears experience distribution:")
print(df['min_years_exp'].value_counts().sort_index().head(15))

exp_desc = df.groupby('domain')['min_years_exp'].describe()
if not exp_desc.empty and len(exp_desc.columns) > 1:
    print(f"\nYears experience by domain:")
    print(exp_desc[['count', 'mean', 'std', 'min', 'max']].round(1))
    
# ===== EXPERIENCE LEVEL (ENHANCED) =====
# Now uses: scraped seniority > years required > education > title keywords
def get_experience_level(row):
    """
    Determine experience level using multiple signals in priority order:
    1. LinkedIn scraped seniority level (most reliable)
    2. Years of experience required from description
    3. Education level required
    4. Title keywords
    """
    # Priority 1: LinkedIn's scraped seniority field
    scraped = str(row.get('detail_seniority', '')).strip().lower()
    if scraped and scraped not in ['', 'nan', 'not applicable']:
        if scraped in ['internship', 'entry level', 'entry']:
            return 'Entry'
        elif scraped in ['associate']:
            return 'Entry'
        elif scraped in ['mid-senior level', 'mid-senior']:
            return 'Mid'
        elif scraped in ['director', 'executive']:
            return 'Senior'
        elif scraped == 'senior':
            return 'Senior'
    
    # Priority 2: Years of experience from description
    years = row.get('min_years_exp')
    if pd.notna(years) and years is not None:
        if years <= 1:
            return 'Entry'
        elif years <= 4:
            return 'Mid'
        else:
            return 'Senior'
    
    # Priority 3: Education level as a proxy for seniority
    edu = row.get('education_required', '')
    if edu == 'PhD':
        return 'Senior'  # PhDs typically go into senior research roles
    
    # Priority 4: Title keywords (last resort)
    title_lower = str(row.get('title', '')).lower()
    if any(kw in title_lower for kw in ['senior', 'sr.', 'lead', 'principal', 'staff', 'director', 'head', 'vp ', 'vice president']):
        return 'Senior'
    if any(kw in title_lower for kw in ['junior', 'jr.', 'entry', 'associate', 'intern', 'graduate', 'new grad']):
        return 'Entry'
    return 'Mid'

df['experience_level'] = df.apply(get_experience_level, axis=1)
df['is_senior'] = df['experience_level'] == 'Senior'
df['is_entry'] = df['experience_level'] == 'Entry'

print(f"\nExperience level distribution:")
print(df['experience_level'].value_counts())
print(f"\nExperience level by domain:")
print(pd.crosstab(df['domain'], df['experience_level']))

# ===== SKILLS EXTRACTION =====
SKILL_KEYWORDS = {
    'Python': ['python'],
    'SQL': ['sql', 'postgresql', 'mysql', 'bigquery', 'redshift'],
    'R': [' r,', ' r.', 'r programming', 'rstudio'],
    'Spark': ['apache spark', 'pyspark', 'spark'],
    'TensorFlow': ['tensorflow', 'tf2', 'tf1'],
    'PyTorch': ['pytorch', 'torch'],
    'Scikit-learn': ['scikit-learn', 'sklearn'],
    'AWS': ['aws ', 'amazon web services', 's3 ', 'ec2 ', 'sagemaker'],
    'Azure': ['azure', 'microsoft azure'],
    'GCP': ['gcp', 'google cloud', 'bigquery', 'vertex ai'],
    'Tableau': ['tableau'],
    'Power BI': ['power bi', 'powerbi'],
    'Snowflake': ['snowflake'],
    'dbt': ['dbt ', 'data build tool'],
    'Airflow': ['airflow', 'apache airflow'],
    'Docker': ['docker'],
    'Kubernetes': ['kubernetes', 'k8s'],
    'LLMs': ['llm', 'large language model', 'gpt', 'chatgpt', 'openai', 'langchain', 'rag'],
    'Git': ['git ', 'github', 'gitlab'],
    'Statistics': ['statistical analysis', 'hypothesis testing', 'a/b test', 'regression'],
    'Excel': ['excel', 'spreadsheet'],
    'Java': ['java '],
    'Scala': ['scala'],
    'NLP': ['nlp', 'natural language processing'],
    'Computer Vision': ['computer vision', 'opencv', 'image recognition'],
}

def extract_skills(row):
    """Extract skills mentioned in the job description."""
    text = ' '.join([
        str(row.get('detail_description', '')),
        str(row.get('description', ''))
    ]).lower()
    
    if not text.strip():
        return ''
    
    found = [skill for skill, patterns in SKILL_KEYWORDS.items()
             if any(p in text for p in patterns)]
    return ', '.join(sorted(found))

df['extracted_skills'] = df.apply(extract_skills, axis=1)

print(f"\nSkill extraction coverage: {(df['extracted_skills'] != '').sum()} / {len(df)}")

# Count skill frequency
all_skills = df['extracted_skills'].str.split(', ').explode()
all_skills = all_skills[all_skills.str.strip() != '']
print(f"\nTop 15 skills:")
print(all_skills.value_counts().head(15))

# Parse dates and basic derived features
df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')
df['title_length'] = df['title'].str.len()
df['has_salary'] = df['salary_mid'].notna()

# Description length — prefer full detail description
desc_len = df['description'].fillna('').str.len()
detail_len = df['detail_description'].fillna('').str.len() if 'detail_description' in df.columns else pd.Series(0, index=df.index)
df['description_length'] = pd.concat([desc_len, detail_len], axis=1).max(axis=1).astype(int)
df['has_full_description'] = detail_len > 100


Education requirement distribution:
education_required
Not Specified         1655
Master's               843
PhD                    407
Bachelor's             374
No Degree Required     260
Name: count, dtype: int64

Education by domain:
education_required  Bachelor's  Master's  No Degree Required  Not Specified  \
domain                                                                        
AI                          85       269                  77            407   
Data Science               289       574                 183           1248   

education_required  PhD  
domain                   
AI                  132  
Data Science        275  

Years experience coverage: 1586 / 3539

Years experience distribution:
min_years_exp
0.0      18
1.0     191
2.0     308
3.0     365
4.0     143
5.0     318
6.0      47
7.0      64
8.0      57
9.0       3
10.0     52
12.0     11
14.0      1
15.0      3
18.0      4
Name: count, dtype: int64

Years experience by domain:
               count

### 9.6 Final Data Quality Report

In [24]:
print("=" * 60)
print("FINAL DATA QUALITY REPORT")
print("=" * 60)
print(f"Total cleaned records: {len(df)}")
print(f"Minimum met (500+): {'✅ YES' if len(df) >= 500 else '⚠️ NO — run more scrapers'}")
print(f"\nSources: {df['source'].nunique()}")
for src, count in df['source'].value_counts().items():
    print(f"  - {src}: {count}")
print(f"\nDomains:")
for dom, count in df['domain'].value_counts().items():
    print(f"  - {dom}: {count}")
print(f"\nIrrelevant jobs filtered during scraping:")
print("  (skipped counts not available when restarting from checkpoint)")
print(f"\nData types present:")
type_map = {
    'Strings': ['title', 'company', 'location', 'source', 'job_category'],
    'Floats': ['salary_min', 'salary_max', 'salary_mid'],
    'Integers': ['title_length', 'description_length'],
    'Booleans': ['has_salary', 'is_senior', 'is_entry'],
    'Dates': ['date_posted'],
    'Categorical': ['domain', 'work_type', 'experience_level', 'state']
}
for dtype, cols in type_map.items():
    present_cols = [c for c in cols if c in df.columns]
    print(f"  - {dtype}: {', '.join(present_cols)}")
print(f"\nMissing values:")
missing = df.isnull().sum()
missing = missing[missing > 0]
for col, count in missing.items():
    print(f"  - {col}: {count} ({count/len(df):.1%})")

FINAL DATA QUALITY REPORT
Total cleaned records: 3539
Minimum met (500+): ✅ YES

Sources: 2
  - LinkedIn: 3434
  - RemoteOK: 105

Domains:
  - Data Science: 2569
  - AI: 970

Irrelevant jobs filtered during scraping:
  (skipped counts not available when restarting from checkpoint)

Data types present:
  - Strings: title, company, location, source, job_category
  - Floats: salary_min, salary_max, salary_mid
  - Integers: title_length, description_length
  - Booleans: has_salary, is_senior, is_entry
  - Dates: date_posted
  - Categorical: domain, work_type, experience_level, state

Missing values:
  - location: 35 (1.0%)
  - salary_min: 3528 (99.7%)
  - salary_max: 3528 (99.7%)
  - salary_text: 3539 (100.0%)
  - tags: 3434 (97.0%)
  - date_posted: 3434 (97.0%)
  - description: 3434 (97.0%)
  - detail_description: 109 (3.1%)
  - detail_seniority: 109 (3.1%)
  - detail_employment_type: 109 (3.1%)
  - detail_job_function: 113 (3.2%)
  - detail_industry: 109 (3.1%)
  - detail_salary: 1435 (4

---
## 10. Export Cleaned Dataset

In [29]:
# Select final columns for export
export_cols = [
    # Core identification
    'title', 'job_category', 'domain', 'ai_signal', 'company', 'location', 'state',
    # Work & experience
    'work_type', 'experience_level', 'education_required', 'min_years_exp',
    'is_senior', 'is_entry',
    # Salary
    'salary_min', 'salary_max', 'salary_mid', 'has_salary',
    # AI detection
    'ai_in_description', 'extracted_skills',
    # Metadata
    'tags', 'date_posted', 'title_length', 'description_length',
    'has_full_description', 'source', 'url',
    # Scraped detail fields
    'detail_seniority', 'detail_employment_type', 'detail_industry',
    'detail_job_function', 'detail_salary', 'detail_work_arrangement',
    'detail_description'
]

df_export = df[[c for c in export_cols if c in df.columns]].copy()

# Strip timezone info for Excel compatibility
if 'date_posted' in df_export.columns:
    df_export['date_posted'] = pd.to_datetime(df_export['date_posted'], errors='coerce')
    if df_export['date_posted'].dt.tz is not None:
        df_export['date_posted'] = df_export['date_posted'].dt.tz_localize(None)

# Save as CSV and Excel
df_export.to_csv('job_listings_cleaned.csv', index=False)
# Strip illegal characters from text columns before Excel export
import re
ILLEGAL_CHARS_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')

def clean_for_excel(val):
    if isinstance(val, str):
        return ILLEGAL_CHARS_RE.sub('', val)
    return val

df_export = df_export.applymap(clean_for_excel)
df_export.to_excel('job_listings_cleaned.xlsx', index=False)

print(f"Exported {len(df_export)} cleaned records")
print(f"  → job_listings_cleaned.csv")
print(f"  → job_listings_cleaned.xlsx")
print(f"\nColumns ({len(df_export.columns)}): {', '.join(df_export.columns)}")
print(f"\nPreview:")
df_export.head()

Exported 3539 cleaned records
  → job_listings_cleaned.csv
  → job_listings_cleaned.xlsx

Columns (33): title, job_category, domain, ai_signal, company, location, state, work_type, experience_level, education_required, min_years_exp, is_senior, is_entry, salary_min, salary_max, salary_mid, has_salary, ai_in_description, extracted_skills, tags, date_posted, title_length, description_length, has_full_description, source, url, detail_seniority, detail_employment_type, detail_industry, detail_job_function, detail_salary, detail_work_arrangement, detail_description

Preview:


,title,job_category,domain,ai_signal,company,location,state,work_type,experience_level,education_required,...,has_full_description,source,url,detail_seniority,detail_employment_type,detail_industry,detail_job_function,detail_salary,detail_work_arrangement,detail_description
0,Data Entry Coordinator,Data (General),Data Science,Data Science,Profound Research,Remote,None,Remote,Entry,Not Specified,...,False,RemoteOK,https://remoteOK.com/remote-jobs/remote-data-entry-coordinator-profound-rese...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Mid Senior AI Cinematic Video Editor,AI (General),AI,AI,EverAI,NaN,None,Unknown,Senior,Not Specified,...,False,RemoteOK,https://remoteOK.com/remote-jobs/remote-mid-senior-ai-cinematic-video-editor...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Business Analyst,Business Analyst,Data Science,Data Science,"Contact Government Services, LLC",NaN,None,Unknown,Mid,Not Specified,...,False,RemoteOK,https://remoteOK.com/remote-jobs/remote-business-analyst-contact-government-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3D Modeling & Python Specialist Freelance AI Trainer Project,AI (General),AI,AI,Invisible Agency,NaN,None,Unknown,Mid,Not Specified,...,False,RemoteOK,https://remoteOK.com/remote-jobs/remote-3d-modeling-python-specialist-freela...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VFX Artist AI Training Vancouver Canada,AI (General),AI,AI,Prolific Academic Ltd,Vancouver,None,Unknown,Mid,Not Specified,...,False,RemoteOK,https://remoteOK.com/remote-jobs/remote-vfx-artist-ai-training-vancouver-can...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## 11. Updated Research Questions for Notebook 2

With full job descriptions now available, the analysis can go deeper:

1. **DS vs AI Demand by City** — What is the ratio of DS to AI roles across major metros? How does this change when we include AI-Adjacent roles (DS titles that mention AI in their descriptions)?

2. **Experience & Education Gap** — What education level (PhD / Master's / Bachelor's) and years of experience do AI roles require vs DS roles? Is the barrier to entry higher for AI?

3. **Geographic Concentration** — Which states dominate hiring for each role category? Is AI hiring more concentrated than DS hiring?

4. **Role Evolution** — How do emerging roles (Prompt Engineer, ML Engineer) compare in volume to established ones (Data Analyst, Business Analyst)?

5. **Skills Demand** — What skills appear most frequently across DS vs AI roles? Which skills are uniquely AI-specific vs shared across both domains?

6. **Hidden AI Demand** — What percentage of DS-titled roles actually require AI/ML skills based on their descriptions? (The 'AI-Adjacent' signal)

### New Columns Available for Analysis

| Column | Description |
|--------|-------------|
| `ai_signal` | 'AI', 'AI-Adjacent', or 'Data Science' — based on title AND description |
| `ai_in_description` | Boolean — description mentions AI/ML even if title doesn't |
| `education_required` | PhD / Master's / Bachelor's / No Degree / Not Specified |
| `min_years_exp` | Minimum years of experience extracted from description |
| `extracted_skills` | Comma-separated list of skills found in description |
| `detail_industry` | Industry scraped from LinkedIn job page |
| `detail_employment_type` | Full-time / Contract / Part-time |
| `work_type` | Remote / Hybrid / On-site (from scraped badge, not just keywords) |
